#**Василенко Иван ИБ N4170**

#**Методы оптимизации**

#**Лабораторная работа 1 Задача ЛП**


| Период (t) | Проект 1 | Проект 2 | Проект 3 | Проект 4 | Проект 5 |
|------------|----------|----------|----------|----------|----------|
| 0          | -195     | -155     | -165     | -285     | -55      |
| 1          | 10       | 45       | 115      | 20       | 10       |
| 2          | 125      | 30       | 125      | 100      | 20       |
| 3          | 85       | 125      | 150      | 200      | 40       |




| Период (t) | Источник 1 | Источник 2 | Источник 3 | Источник 4 | Источник 5 |
|------------|------------|------------|------------|------------|------------|
| 0          | 85         | 135        | 160        | 120        | 185        |
| 1          | -90        | -40        | -95        | -100       | -10        |
| 2          | -105       | -120       | -75        | -50        | -130       |
| 3          | -10        | -110       | -45        | -135       | -135       |

###**Математическая модель**

$\sum_{j=1}^{5} x_j \cdot \mathrm{NPV}_j + \sum_{k=1}^{5} y_k \cdot \mathrm{NPV}_k$ -> max

где $\mathrm{NPV}_j = \sum_{t=0}^{3} P_{j,t}$ (положительная для проектов, отражает прибыль), $\mathrm{NPV}_k = \sum_{t=0}^{3} Q_{k,t}$ (отрицательная для источников, отражает стоимость финансирования).

S - стартовый капитал в период t=0

######**Целевая функция:**

$\, 25x_1 + 45x_2 + 225x_3 + 35x_4 + 15x_5 -120y_1 -135y_2 -55y_3 -165y_4 -90y_5$ -> max

######**Ограничения:**

- Для t=0: $-195x_1 -155x_2 -165x_3 -285x_4 -55x_5 + 85y_1 +135y_2 +160y_3
+120y_4 +185y_5\geq -S$
- Для t=1: $10x_1 +45x_2 +115x_3 +20x_4 +10x_5 -90y_1 -40y_2 -95y_3 -100y_4 -10y_5 \geq 0$
- Для t=2: $125x_1 +30x_2 +125x_3 +100x_4 +20x_5 -105y_1 -120y_2 -75y_3 -50y_4 -130y_5 \geq 0$
- *Для* t=3: $85x_1 +125x_2 +150x_3 +200x_4 +40x_5 -10y_1 -110y_2 -45y_3 -135y_4 -135y_5 \geq 0$

###**Код**

In [ ]:
!pip install pulp
import numpy as np
import scipy.optimize as opt
import pulp
import tracemalloc
import time
import resource

In [ ]:
# Стартовый капиатл в период t=0
S = 100.0

# P - денежные потоки проектов: строки — проекты, столбцы — периоды
P = np.array([
    [-195.0, 10.0, 125.0, 85.0],
    [-155.0, 45.0, 30.0, 125.0],
    [-165.0, 115.0, 125.0, 150.0],
    [-285.0, 20.0, 100.0, 200.0],
    [-55.0, 10.0, 20.0, 40.0]
])

# Q - денежные потоки источников: строки — источники, столбцы — периоды
Q = np.array([
    [85.0, -90.0, -105.0, -10.0],
    [135.0, -40.0, -120.0, -110.0],
    [160.0, -95.0, -75.0, -45.0],
    [120.0, -100.0, -50.0, -135.0],
    [185.0, -10.0, -130.0, -135.0]
])

In [ ]:
# Расчёт NPV для проектов - сумма по столбцам в P
NPV_p = np.sum(P, axis=1)

# Расчёт NPV для источников -сумма по столбцам в Q
NPV_q = np.sum(Q, axis=1)

# Вектор коэффициентов целевой функции - объединение NPV проектов и источников
c = np.concatenate((NPV_p, NPV_q))

# Матрица A коэффициентов ограничений из неравенств
A = np.zeros((4, 10))

for t in range(4):
    A[t, :5] = -P[:, t]

    A[t, 5:] = -Q[:, t]

# Вектор правых частей ограничений
b = np.array([S, 0.0, 0.0, 0.0])

# Определение границ для переменных: все переменные от 0 до 1
bounds = [(0, 1) for _ in range(10)]

In [ ]:
# SciPy
start_time = time.time()
tracemalloc.start()

# Вызов функции linprog: максимизация (поэтому -c), с неравенствами A_ub x <= b_ub, границами и методом 'highs'
res = opt.linprog(-c, A_ub=A, b_ub=b, bounds=bounds, method='highs')

# Вычисление оптимального значения целевой функции (поскольку максимизировали -c, то -res.fun)
obj_scipy = -res.fun

mem_scipy, mem_scipy_max = tracemalloc.get_traced_memory()
tracemalloc.stop()
end_time = time.time()

time_scipy = end_time - start_time

In [ ]:
# PuLP
# Создание объекта проблемы оптимизации: максимизация
prob = pulp.LpProblem("Finance_Opt", pulp.LpMaximize)

# Создание переменных x_j от 0 до 1 для проектов
x_vars = [pulp.LpVariable(f"x_{j}", 0, 1) for j in range(5)]

# Создание переменных y_k от 0 до 1 для источников
y_vars = [pulp.LpVariable(f"y_{k}", 0, 1) for k in range(5)]

# Целевая функция
prob += sum(NPV_p[j] * x_vars[j] for j in range(5)) + sum(NPV_q[k] * y_vars[k] for k in range(5))

# Добавление ограничений
for t in range(4):
    # Вычисление cash flow в период t: сумма потоков проектов * x + потоков источников * y
    flow = sum(P[j, t] * x_vars[j] for j in range(5)) + sum(Q[k, t] * y_vars[k] for k in range(5))

    # Добавление ограничения: flow >= -S для t=0, иначе >= 0
    prob += flow >= - (S if t == 0 else 0)

start_time = time.time()
tracemalloc.start()

# Решение проблемы с солвером CBC
prob.solve(pulp.PULP_CBC_CMD(msg=False))

# Получение оптимального значения целевой функции
obj_pulp = pulp.value(prob.objective)

mem_pulp, mem_pulp_max = tracemalloc.get_traced_memory()
tracemalloc.stop()
end_time = time.time()

time_pulp = end_time - start_time

In [ ]:
# Вывод результатов SciPy
print("SciPy Results:")
print(f"Optimal objective value: {obj_scipy}")
print("Optimal project fractions (x):")
for i in range(5):
    print(f"x_{i}: {res.x[i]}")
print("Optimal source fractions (y):")
for i in range(5):
    print(f"y_{i}: {res.x[i+5]}")
print(f"Time taken: {time_scipy} seconds")
print(f"Memory used: {mem_scipy_max / 1024:.2f} KB")

SciPy Results:
Optimal objective value: 202.65625
Optimal project fractions (x):
x_0: 0.0
x_1: 0.0
x_2: 1.0
x_3: 0.0
x_4: 0.0
Optimal source fractions (y):
y_0: 0.0
y_1: 0.0
y_2: 0.40625
y_3: 0.0
y_4: 0.0
Time taken: 0.008479833602905273 seconds
Memory used: 23.95 KB


In [ ]:
# Вывод результатов PuLP
print("\nPuLP Results:")
print(f"Optimal objective value: {obj_pulp}")
print("Optimal project fractions (x):")
for j in range(5):
    print(f"x_{j}: {pulp.value(x_vars[j])}")
print("Optimal source fractions (y):")
for k in range(5):
    print(f"y_{k}: {pulp.value(y_vars[k])}")
print(f"Time taken: {time_pulp} seconds")
print(f"Memory used: {mem_pulp_max / 1024:.2f} KB")


PuLP Results:
Optimal objective value: 202.65625
Optimal project fractions (x):
x_0: 0.0
x_1: 0.0
x_2: 1.0
x_3: 0.0
x_4: 0.0
Optimal source fractions (y):
y_0: 0.0
y_1: 0.0
y_2: 0.40625
y_3: 0.0
y_4: 0.0
Time taken: 0.01078033447265625 seconds
Memory used: 58.89 KB
